In [1]:
# With much thanks to Islam S. for identifying that there was a missing import!
import os, re, math
from pathlib import Path
from datetime import datetime

import dotenv, yaml
dotenv.load_dotenv("configs/local.env")


def model_cache_path(model_id):
    home_dir = Path.home()
    cache_dir = os.environ.get('HF_HUB_CACHE',  home_dir / ".cache" / "huggingface" / "hub")

    model_hf = Path(cache_dir) / ("models--" + model_id.replace("/", "--"))
    model_ref = (model_hf / "refs" / "main").read_text(encoding="utf-8").strip()
    model_path = model_hf / "snapshots" / model_ref

    return model_path

In [2]:
DATASET_NAME = "ed-donner/pricer-data"
#BASE_MODEL = "meta-llama/Llama-3.1-8B"
BASE_MODEL = "meta-llama/Llama-3.2-1B"
# BASE_MODEL = "data/huggingface/hub/models--meta-llama--Llama-3.2-3B/snapshots/13afe5124825b4f3751f836b40dafda64c1ed062"
# BASE_MODEL = "data/huggingface/hub/models--meta-llama--Llama-3.2-1B/snapshots/4e20de362430cd3b72f300e6b0f18e50e7166e08"

PROJECT_NAME = "pricer"

#RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
RUN_NAME = "2025-08-11"

# Run name for saving the model in the hub
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_ID = f"{os.environ['HF_USER']}/{PROJECT_RUN_NAME}"

In [3]:
# Optonal: offline mode

os.environ['HF_HUB_OFFLINE'] = 'True'
os.environ['WANDB_MODE'] = 'offline'
BASE_MODEL = model_cache_path(BASE_MODEL)

In [4]:
#from huggingface_hub import login
import wandb
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt

#login(os.environ['HF_TOKEN'], add_to_git_credential=True)

# Configure Weights & Biases to record against our project
wandb.init(project=PROJECT_NAME, name=RUN_NAME)
# wandb.login(key=os.environ['WANDB_API_KEY'], force=True)

os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" # if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

In [5]:
from tqdm import tqdm
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

⚙️  Running in WANDB offline mode


In [7]:
dataset = load_dataset(DATASET_NAME)
train_dataset, temp_dataset = dataset['train'], dataset['test']

eval_test_split = temp_dataset.train_test_split(test_size=0.5, seed=42)
eval_dataset = eval_test_split["train"]  # 验证集
test_dataset = eval_test_split["test"]   # 测试集

print("==> train_dataset[0]:", train_dataset[0])
print("==> test_dataset[0]:", test_dataset[0])

Using the latest cached version of the dataset since ed-donner/pricer-data couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at data/huggingface/datasets/ed-donner___pricer-data/default/0.0.0/f38df5f756eafe7e331348b26a823e4d706f4a08 (last modified on Thu Aug 14 12:19:34 2025).


==> train_dataset[0]: {'text': 'How much does this cost to the nearest dollar?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7\n\nPrice is $227.00', 'price': 226.95}
==> test_dataset[0]: {'text': 'How much does this cost to the nearest dollar?\n\nMatte Black Hoop Drop Step Side Nerf Bars Rail Running Boards For 07-18 Chevy Silverado/GMC Sierra Crew Cab\

In [8]:
eval_dataset = eval_dataset.map(lambda x: {"text": f"{x['text']}{x['price']}" })
print("==> eval_dataset[0]:", eval_dataset[0])

==> eval_dataset[0]: {'text': "How much does this cost to the nearest dollar?\n\nKira Home Atlantic 16 Modern Chandelier, Metal Wavy Design Drum Shade, White Finish\nProduct Overview Kira Home's Atlantic 16 hanging chandelier adds a modern, but rustic touch to any home. Versatile and understated, this fixture is crafted of a unique, metal drum shade in a beautiful white finish. The classic shape of Atlantic makes it suitable for varying interiors while its stunning wave pattern makes it the perfect statement piece. Enhance your existing furnishings and decor styles such as traditional, craftsman and classic. Well constructed and durable, hang this ceiling lamp in many areas in the home. Make an elegant impression over a kitchen island. Suspend over a dining room table, in a game room over a pool table or in an office lobby. Adjust hanging heights as the adjustable down\n\nPrice is $44.99", 'price': 44.99}


In [9]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
# quant_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

#print("named_modules:", list(base_model.named_modules()))
#for name, module in base_model.named_modules():
#    if any(x in name for x in ["proj", "fc"]):
#        print(name)

Memory footprint: 1.0 GB


In [11]:
# Train parameters

# Hyperparameters for QLoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# Hyperparameters for Training
EPOCHS = 2 # you can do more epochs if you wish, but only 1 is needed - more is probably overkill
BATCH_SIZE = 4 # on an A100 box this can go up to 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Admin config - note that SAVE_STEPS is how often it will upload to the hub
# I've changed this from 5000 to 2000 so that you get more frequent saves
STEPS = 50
SAVE_STEPS = 2000

In [12]:
# First, specify the configuration parameters for LoRA
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [13]:
# Next, specify the general configuration parameters for training

output_dir = Path("data") / PROJECT_RUN_NAME

train_parameters = SFTConfig(
    output_dir=output_dir,
    model_init_kwargs={"torch_dtype": "auto"},
    max_length=1024,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    #eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    logging_steps=STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True, # False for cpu
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb", # if LOG_TO_WANDB else None; "tensorboard"
    run_name=RUN_NAME,
    dataset_text_field="text",
    save_strategy="steps",
    #hub_strategy="every_save",
    #push_to_hub=True,
    #hub_model_id=HUB_MODEL_ID,
    #hub_private_repo=True,

    completion_only_loss=True,

    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    restore_callback_states_from_checkpoint=True,
)

In [14]:
# And now, the Supervised Fine Tuning Trainer will carry out the fine-tuning
# Given these 2 sets of configuration parameters
# The latest version of trl is showing a warning about labels - please ignore this warning
# But let me know if you don't see good training results (loss coming down).

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    peft_config=lora_parameters,
    args=train_parameters,

    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

/home/hello/Apps/llm01/.venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:389: UserWarning: You passed model_init_kwargs to the `SFTConfig`, but your model is already instantiated. The `model_init_kwargs` will be ignored.
  warnings.warn(


Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [15]:
last_ckpt = get_last_checkpoint(output_dir)

print("==> fine_tuning start: {}, {}".format(datetime.now().astimezone().strftime("%FT%T%:z"), last_ckpt))

# Fine-tune!
fine_tuning.train(resume_from_checkpoint=last_ckpt if last_ckpt else None)

print("<== fine_tuning end:", datetime.now().astimezone().strftime("%FT%T%:z"))

==> fine_tuning start: 2025-08-14T12:21:08+08:00, None


wandb: WARNING URL not available in offline run


Step,Training Loss,Validation Loss
2000,2.625000,2.754052
4000,2.602100,2.657531
6000,2.462900,2.596331
8000,2.440200,2.513240
10000,2.406100,2.470858
12000,2.301400,2.454391
14000,2.396100,2.434101
16000,2.318600,2.414668
18000,2.343300,2.394873
20000,2.335500,2.386945


wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-2000)... Done. 0.4s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-4000)... Done. 0.4s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-6000)... Done. 0.3s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-8000)... Done. 0.3s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-10000)... Done. 0.4s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-12000)... Done. 0.4s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-14000)... Done. 0.4s
wandb: WAR

<== fine_tuning end: 2025-08-14T17:16:44+08:00


In [15]:
fine_tuning.save_model(output_dir)

# Push our fine-tuned model to Hugging Face
#fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
#print(f"Saved to the hub: {PROJECT_RUN_NAME}")

wandb.finish()

eval/loss,█▁
eval/mean_token_accuracy,█▁
eval/num_tokens,▁█
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▃▄▁▂▂▂▃▃▅▅▁▃▃▂▃▄▁▂▄█▄▂▄▁▃▃▂▃▇▂▂▂▂▃▂▂▄▂▆▃
train/learning_rate,███▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
train/loss,▁▃▇▅▆▆▅▅▅▄▅▃▅▅▇▇▅▆▆▃▄▅▅▇▄▃▄▃▄▇▅▇█▅▄▂▃▅▅▂
